I've recently been learning to use Spark for work and I've found it really interesting. Spark is an open-source framework used for processing large amounts of data quickly and reliably. To showcase some of the basic operations in Spark, I've written a brief tutorial to analyse some data about Pokémon.

Spark is written in Scala, but there are several supported APIs in different languages, including Java, Scale, Python and R. For this tutorial I'll be using the Python Spark API, known as PySpark. 

## Processing Data

Before we go into the details of using Spark's Python API, let's take a step back and look at some of the issues that Spark was designed to solve.

When processing data a number of transformations are often applied in order to get the data into a form that is useful. This can include transformations like filtering, grouping and mapping.

For example, take a list of Pokémon. Each Pokémon has a list of attributes, like HP, attack and its type. We can do a bunch of transformations with this data, such as finding the Pokémon with the highest HP or working out the average attack for different Pokémon types.

In a system where only one computer is used processing this data might look like this:

![Data Flow](images/spark-intro/etl_1.svg)


Data is fed in at one end (in this case data at Pokémon), a bunch of transformations are applied to the data in the middle, and then the data is output at the other end (such as a report or another database table). This system is often referred to as Extract, Transform, Load (ETL).

There are a few limitations to this approach of using a single machine to process large datasets:
1. Data is often processed synchronously. For large sets of data this can be slow as records as processed one at a time.
1. When data is processed in parallel or asynchronously, this can lead to complex code or unforseen bugs like race conditions.
1. The system is not very fault tolerant. If the single computer crashes, the progress applying transformations to the data is lost. 

To overcome these issues, Spark uses a techique called Resilient Distributed Datasets (RDD).

### Resilient Distributed Datasets

As with a standard ETL system, Spark takes a data input, transforms the data, and then loads it to another destination. The most significant difference is that Spark does transformation across a cluster of computers, instead of just one.

To process the data across a cluster of computers, Spark breaks the datasets into chunks. This allows the data to be processed in parallel on several machines, after which it is recombined into a single dataset. This technique is called a Resilient Distributed Dataset (RDD) and is fundamental to how Spark works.

Here is a modified version of the ETL diagram that illustrates data chunking and distribution:

![Data Flow](images/spark-intro/rdd_1.svg)

By chunking and distributing data in this way, Spark is able to process large datasets very quickly. It also means that it can handle the amount of data increasing by adding more computers to the cluster.

In addition to chunking and distributing data, Spark also does one other thing to the data in an RDD: it duplicates it. Before distributing the data across the cluster, Spark duplicates each chunk. The reason it does this is to protect against computers in the cluster failing. If a computer crashes while it is processing a chunk of data, it doesn't matter, the same chunk of data can be sent to be processed by another healthy computer. 

By duplicating data with RDDs, the system is much more resilient to failures.


### Dataframes

Now that we've covered the concepts behind processing data with Spark's RDDs, lets look at how to use them.

Spark uses Dataframes to manage data. Dataframes are structured like a spreadsheet: they have a rows for data items and columns for each attribute of the data. Transformations can be applied to data directly from the dataframe using a variety of methods.

Dataframes use RDD features under the hood. This means that the transformations applied to dataframes can be done across a cluster of computers, taking advantage of the speed and reslience of RDDs. Spark manages all of this complexity, allowing the programmer to define how they want the data to be processed, without worrying about distributing the data across a cluster of computers.

As well as running Spark on a cluster, we're able to run Spark on a single computer without any significant differences to the code. This means that you can develop your Spark code locally on a single computer before you run it on a cluster in production.

Let's take a look at using Dataframes in PySpark.

## PySpark

For this example I'll be using Pokémon data stored in a json file. This data was retrieved using the [Pokémon API](https://pokeapi.co/) and I did some additional processing to extract only the attributes that I wanted. The data has several columns, including the base stats for each Pokémon as well as it's name, types, and weight and height. I've made [the data available to download](/static/spark-intro/pokemon.json) if you want to follow along with the example.


To get started with Spark's Python API you need to install PySpark:

``` bash
pip install pyspark
```

Once `pyspark` is successfully installed we can start using Spark to process data.

First we need to import pyspark and set up a spark session:


In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('pokemon').getOrCreate()

This will start a new Spark app, or get an existing one if it already exists. I've named my Spark app `pokemon` in this case.

Now that I've started the session I can create a dataframe with the Pokémon data:

In [2]:
pokemon = spark.read.json('static/spark-intro/pokemon.json')

Spark comes with several methods to read common file types. Here I've used the `json()` read method. There are also methods for csv and text files, among many others.

Now that the data is loaded into a dataframe, we can see what it looks like using the `show()` method:

In [3]:
pokemon.show()

+------+-------+------+---+----------+-----+--------------+---------------+-----+------+------+------+
|attack|defense|height| hp|      name|order|special-attack|special-defense|speed|type_1|type_2|weight|
+------+-------+------+---+----------+-----+--------------+---------------+-----+------+------+------+
|    49|     49|     7| 45| bulbasaur|    1|            65|             65|   45| grass|poison|    69|
|    62|     63|    10| 60|   ivysaur|    2|            80|             80|   60| grass|poison|   130|
|    82|     83|    20| 80|  venusaur|    3|           100|            100|   80| grass|poison|  1000|
|    52|     43|     6| 39|charmander|    5|            60|             50|   65|  fire|  null|    85|
|    64|     58|    11| 58|charmeleon|    6|            80|             65|   80|  fire|  null|   190|
|    84|     78|    17| 78| charizard|    7|           109|             85|  100|  fire|flying|   905|
|    48|     65|     5| 44|  squirtle|   10|            50|             6

We can also quickly query the number of records in the data using the `.count()` method:

In [4]:
pokemon.count()

949

We can also view the schema for the table to check the data types that Spark has inferred for each column, using the `.printSchema()` method:

In [5]:
pokemon.printSchema()

root
 |-- attack: long (nullable = true)
 |-- defense: long (nullable = true)
 |-- height: long (nullable = true)
 |-- hp: long (nullable = true)
 |-- name: string (nullable = true)
 |-- order: long (nullable = true)
 |-- special-attack: long (nullable = true)
 |-- special-defense: long (nullable = true)
 |-- speed: long (nullable = true)
 |-- type_1: string (nullable = true)
 |-- type_2: string (nullable = true)
 |-- weight: long (nullable = true)



If I only want to select specific columns I can do this using the `.select()` method:

In [6]:
pokemon.select('name', 'type_1').show()

+----------+------+
|      name|type_1|
+----------+------+
| bulbasaur| grass|
|   ivysaur| grass|
|  venusaur| grass|
|charmander|  fire|
|charmeleon|  fire|
| charizard|  fire|
|  squirtle| water|
| wartortle| water|
| blastoise| water|
|  caterpie|   bug|
|   metapod|   bug|
|butterfree|   bug|
|    weedle|   bug|
|    kakuna|   bug|
|  beedrill|   bug|
|    pidgey|normal|
| pidgeotto|normal|
|   pidgeot|normal|
|   rattata|normal|
|  raticate|normal|
+----------+------+
only showing top 20 rows



Let's run through some common transformations on the sales data.



### Filtering

Filtering can be used to quickly return a subset of the data set. Here I am using the `.filter()` method to retrieve all of the Pokémon with the `'dragon'` type:

In [7]:
pokemon.filter(pokemon.type_1 == 'dragon') \
    .select('name', 'type_1') \
    .show()

+---------+------+
|     name|type_1|
+---------+------+
|  dratini|dragon|
|dragonair|dragon|
|dragonite|dragon|
|  altaria|dragon|
|    bagon|dragon|
|  shelgon|dragon|
|salamence|dragon|
|   latias|dragon|
|   latios|dragon|
| rayquaza|dragon|
|    gible|dragon|
|   gabite|dragon|
| garchomp|dragon|
|     axew|dragon|
|  fraxure|dragon|
|  haxorus|dragon|
|druddigon|dragon|
| reshiram|dragon|
|   zekrom|dragon|
|   kyurem|dragon|
+---------+------+
only showing top 20 rows



When filtering I can use multiple conditions. The Pokémon can have two type attributes, `type_1` and `type_2`. Using the  `|` (or) operator we can filter data when either operator is true. In this case I'm filtering the Pokémon that have the `'fire'` value in `type_1` or `type_2`:

In [8]:
pokemon \
    .filter((pokemon.type_1 == 'fire') | (pokemon.type_2 == 'fire')) \
    .select('name', 'type_1', 'type_2') \
    .show()

+----------+------+------+
|      name|type_1|type_2|
+----------+------+------+
|charmander|  fire|  null|
|charmeleon|  fire|  null|
| charizard|  fire|flying|
|    vulpix|  fire|  null|
| ninetales|  fire|  null|
| growlithe|  fire|  null|
|  arcanine|  fire|  null|
|    ponyta|  fire|  null|
|  rapidash|  fire|  null|
|    magmar|  fire|  null|
|   flareon|  fire|  null|
|   moltres|  fire|flying|
| cyndaquil|  fire|  null|
|   quilava|  fire|  null|
|typhlosion|  fire|  null|
|    slugma|  fire|  null|
|  magcargo|  fire|  rock|
|  houndour|  dark|  fire|
|  houndoom|  dark|  fire|
|     magby|  fire|  null|
+----------+------+------+
only showing top 20 rows



Each transformation method of a `Dataframe` returns a new `Dataframe`. If I want to reuse the result of a tranformation I can store the result in a variable. Here I want to find the number of Pokémon that have a height attribute higher than 20:

In [9]:
tall_pokemon = pokemon.filter(pokemon.height > 20)

Since I stored the result of the transformation in a variable I can reuse it. First let's see the number of tall Pokémon:

In [10]:
tall_pokemon.count()

90

I can sort the data using the `.sort()` method to order the data. The `.height.desc()` part of the code determines that the sort by the `height` attribute in descending order:

In [11]:
tallest_pokemon = tall_pokemon \
    .sort(pokemon.height.desc()) \
    .first()
    
tallest_pokemon

Row(attack=90, defense=45, height=145, hp=170, name='wailord', order=406, special-attack=90, special-defense=45, speed=60, type_1='water', type_2=None, weight=3980)

Here I've used the `.first()` method to retrieve the first row in the dataset. This returns a `Row` object, that has an attribute for each column:

In [12]:
tallest_pokemon.name

'wailord'

### Aggregations and Calculations

Quite often when processing data you may want to aggregate several rows together into a single value, like a sum or an average. 

Spark provides these featues in variety of different ways. 

I want to see which Pokémon types are in the data:

In [13]:
pokemon.select('type_1') \
    .distinct() \
    .show()

+--------+
|  type_1|
+--------+
|     bug|
|  normal|
|   ghost|
|   grass|
|   steel|
|     ice|
|   water|
|  ground|
|  flying|
|   fairy|
|    dark|
|fighting|
|  dragon|
|  poison|
| psychic|
|    rock|
|electric|
|    fire|
+--------+



I tell Spark which columns I want to include using the `select()` method. Then I use the `distinct()` method to remove duplicate values from the result. 

Next I want to see how tall the Pokémon would be if I stacked them on top of one another:

In [14]:
from pyspark.sql.functions import sum

pokemon.select(sum('height')) \
    .show()

+-----------+
|sum(height)|
+-----------+
|      11656|
+-----------+



I gave the column name to the `sum()` method to calculate the total height. This is called from inside of the `select()` method to return only the calculated value as a new column.

I can also see the average Pokémon base `hp` using the `avg()` method:

In [15]:
from pyspark.sql.functions import avg

pokemon.select(avg('hp')) \
    .show()

+-----------------+
|          avg(hp)|
+-----------------+
|68.95468914646997|
+-----------------+



There are several functions that can compute aggregates like this. Here I'm using the `max()` function to find the Pokémon with the highest base `hp`:

In [16]:
from pyspark.sql.functions import max

pokemon.select(max('hp')) \
    .show()

+-------+
|max(hp)|
+-------+
|    255|
+-------+



### Grouping

Finally, let's look at grouping data. Using the `groupBy()` method I can consolidate data into related groups. After I have grouped the data I can perform aggregations and other transormations on the data.

In this example I want to see the max `hp` for each Pokémon type. I use the `groupBy()` method to group the data by Pokémon type:

In [17]:
from pyspark.sql.functions import round, desc

pokemon_types = pokemon.groupBy(pokemon['type_1'])

pokemon_types.max() \
    .select('type_1', 'max(hp)') \
    .sort(desc('max(hp)')) \
    .show()

+--------+-------+
|  type_1|max(hp)|
+--------+-------+
|  normal|    255|
|    dark|    223|
|  dragon|    216|
| psychic|    190|
|   water|    170|
|   ghost|    150|
|fighting|    144|
|   fairy|    126|
|   grass|    123|
|    rock|    123|
|  ground|    115|
|    fire|    115|
|     ice|    110|
|     bug|    107|
|  poison|    105|
|   steel|    100|
|electric|     90|
|  flying|     85|
+--------+-------+



After I have grouped the data by country, I get the max value for all the columns using the `.max()` method. I then select only the columns I want to display and sort them into descending order.

Applying transformations to columns can give them long and cumbersome names. The above example isn't that bad as it only renames the `hp` attribute to `max(hp)`. In most cases we'd want to rename this.

In this final example I want to find the average attack for each Pokémon group:

In [18]:
pokemon_types.avg() \
    .select('type_1', round('avg(attack)', 2).alias('avg attack')) \
    .sort(desc('avg attack')) \
    .show()

+--------+----------+
|  type_1|avg attack|
+--------+----------+
|  dragon|    108.67|
|fighting|      99.1|
|  ground|     96.22|
|   steel|     93.13|
|    rock|     89.98|
|    dark|      84.7|
|    fire|     84.25|
|  flying|     78.75|
|   ghost|      76.3|
|  normal|     75.86|
|   grass|     75.15|
|   water|     74.71|
|  poison|     73.54|
|     ice|     73.07|
| psychic|     72.55|
|     bug|     72.49|
|electric|     68.15|
|   fairy|     61.21|
+--------+----------+



I've used the `round()` function to round the average attack to two decimal places. I use the `.alias()` method to rename the column to `avg attack`. If I didn't rename the column its name would be `'round(avg(attack), 2)'`.

# Summary


Spark is an excellent tool for processing data. Using computing clusters it can process large amounts of data relatively quickly. It's API are also very level, handling the complex cluster management itself, leaving the programmer to focus on writing business logic. 

I find Spark a very interesting and powerful tool. In this tutorial I've covered a broad range of uses for Spark's dataframes, including filtering, aggregations and grouping. From these short example you can see that with only a few lines of code it's possible to write powerful, yet readable, code for processing data.